
# GraphRAG Fundamentals with Neo4j Aura, Serper and GPT-OSS

## Learning Objectives

In this notebook you will learn:

- What GraphRAG is
- How GraphRAG differs from traditional RAG
- How to collect data using Serper
- How to transform text into entities and relationships
- How Neo4j stores a knowledge graph
- How hybrid retrieval combines graphs and vectors
- How to answer questions using GraphRAG

## Basic RAG

Question → Vector Search → Relevant Chunks → LLM → Answer

## GraphRAG

Question → Graph Retrieval + Vector Retrieval → LLM → Answer

GraphRAG excels when relationships between entities matter.


## Environment Setup

### What are we trying to achieve?
Install and configure the libraries required for GraphRAG. We need LangChain, Neo4j integration, Ollama integration and Serper search.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [5]:
#!pip install -U langchain langchain-community langchain-core langchain-experimental langchain-neo4j langchain-ollama langchain-text-splitters neo4j python-dotenv google-search-results tiktoken pydantic


## Load Configuration

### What are we trying to achieve?
Load environment variables such as Neo4j Aura credentials and Serper API key from the .env file.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")


## Verify Environment

### What are we trying to achieve?
Print configuration values to confirm that the notebook can see the required credentials.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [7]:
print(NEO4J_URI)
print(NEO4J_USERNAME)
print(NEO4J_PASSWORD)
print(SERPER_API_KEY)

neo4j+s://p-mt-2e58c6e6c448-1-0081.production-orch-0703.neo4j.io
a6d250de
JZ6B_6UD9lRjdTJSGwIpm3Yn_1ddz784vzsBJm5SPtU
1ca81cd07f35e2647405477da0f55fa9340ff9de


## Validate Neo4j Connectivity

### What are we trying to achieve?
Before building GraphRAG, verify that Neo4j Aura is reachable and authentication works.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [8]:
from neo4j import GraphDatabase

URI = "neo4j+s://p-mt-2e58c6e6c448-1-0081.production-orch-0703.neo4j.io"

driver = GraphDatabase.driver(
    URI,
    auth=(
        "a6d250de",
        "JZ6B_6UD9lRjdTJSGwIpm3Yn_1ddz784vzsBJm5SPtU"
    )
)

driver.verify_connectivity()

print("Connected!")

Connected!


## Install Neo4j Integration

### What are we trying to achieve?
Use the newer langchain-neo4j package rather than older community implementations.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [9]:
#!pip install -U langchain-neo4j

## Import GraphRAG Components

### What are we trying to achieve?
Import document loaders, text splitters, graph transformers, Neo4j integration and Ollama models.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [10]:
from langchain_core.documents import Document
from langchain_text_splitters import TokenTextSplitter

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.graphs import Neo4jGraph
#from langchain_community.vectorstores import Neo4jVector
from langchain_neo4j import Neo4jVector

from langchain_experimental.graph_transformers import LLMGraphTransformer

from langchain_ollama import ChatOllama, OllamaEmbeddings


d:\RAG Practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sr43993\AppData\Local\Temp\ipykernel_9184\4152653209.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper
C:\Users\sr43993\AppData\Local\Temp\ipykernel_9184\4152653209.py:9: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


## Configure LLM and Embeddings

### What are we trying to achieve?
GPT-OSS performs reasoning and graph extraction. Nomic embeddings power vector retrieval.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [11]:
llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0
)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


## Create Neo4j Graph Connection

### What are we trying to achieve?
Create a LangChain graph object that will store entities and relationships.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [12]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

graph


C:\Users\sr43993\AppData\Local\Temp\ipykernel_9184\3044172980.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


## Alternative Aura Connection

### What are we trying to achieve?
This cell uses the routing hostname that worked successfully with Aura in the corporate environment.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [13]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url="neo4j+s://p-mt-2e58c6e6c448-1-0081.production-orch-0703.neo4j.io",
    username="a6d250de",
    password="JZ6B_6UD9lRjdTJSGwIpm3Yn_1ddz784vzsBJm5SPtU",
    database="a6d250de",
)

print(graph.query("RETURN 1 AS test"))

[{'test': 1}]


## Corporate SSL Fixes

### What are we trying to achieve?
Optional fixes for corporate laptops that use SSL inspection tools such as Cisco Umbrella.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [14]:
#! pip install python-certifi-win32

## Additional Certificate Support

### What are we trying to achieve?
Optional package for integrating system certificates into Python.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [15]:
#! pip install pip-system-certs

## Collect Data Using Serper

### What are we trying to achieve?
Retrieve web content that will become the source material for GraphRAG.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [16]:
search = GoogleSerperAPIWrapper()

query = "Elizabeth I"

results = search.results(query)

len(results.get("organic", []))


9

## Convert Search Results to Documents

### What are we trying to achieve?
Transform search results into LangChain Document objects.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [17]:
docs = []

for item in results.get("organic", []):
    docs.append(
        Document(
            page_content=f"""
Title: {item.get('title','')}

{item.get('snippet','')}

Source: {item.get('link','')}
""",
            metadata={
                "source": item.get("link","")
            }
        )
    )

len(docs)


9

## Chunk the Documents

### What are we trying to achieve?
Split large documents into smaller chunks suitable for LLM processing.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [18]:
text_splitter = TokenTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

documents = text_splitter.split_documents(docs)

len(documents)


9

## Build the Knowledge Graph

### What are we trying to achieve?
LLMGraphTransformer extracts entities and relationships from text chunks.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [19]:
llm_transformer = LLMGraphTransformer(
    llm=llm
)

graph_documents = llm_transformer.convert_to_graph_documents(
    documents
)

len(graph_documents)


9

## Store Graph in Neo4j

### What are we trying to achieve?
Persist nodes and relationships into Neo4j Aura.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [20]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True,
)


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=257, offset=256>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 256, 'line': 1, 'column': 257}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MERGE (d:Document {id:$document.metadata.id}) SET d.text = $document.page_content SET d += $document.metadata WITH d UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties MERGE (d)-[:MENTIONS]->(source) WITH source, row CALL apoc.create.addLabels( source, [row.type] 

## Create Hybrid Vector Index

### What are we trying to achieve?
Create a vector index on top of graph data for hybrid retrieval.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [21]:
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)


## Create Retriever

### What are we trying to achieve?
Expose the vector index as a retriever that can answer questions.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [22]:
retriever = vector_index.as_retriever(
    search_kwargs={"k": 5}
)


## Ask Questions

### What are we trying to achieve?
Retrieve relevant information and inspect the results.

### Why is this important?
This step is a building block of the complete GraphRAG pipeline.


In [23]:
question = "Who succeeded Elizabeth I?"

results = retriever.invoke(question)

for doc in results:
    print(doc.page_content[:1000])
    print("-" * 80)


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL


text: Title: Who succeeded Elizabeth I of England? - Quora

Content: She was succeeded by King James VI of Scotland, who became King James I of England. Queen Elizabeth II succeeded her father, King George VI, in ...

Source: https://www.quora.com/Who-succeeded-Elizabeth-I-of-England
--------------------------------------------------------------------------------

text: Title: Elizabeth I's Heir? : r/Tudorhistory - Reddit

Content: As we know, James VI & I ultimately succeeded Elizabeth I in 1603 and presided over the Union of the Crowns.

Source: https://www.reddit.com/r/Tudorhistory/comments/1c9cpjo/elizabeth_is_heir/
--------------------------------------------------------------------------------

text: Title: Who Really Deserved the Throne After Elizabeth I? - YouTube

Content: Did the crown pass peacefully to from the Tudors to the Stuarts when Elizabeth I died? The story goes that she finally named an heir on her ...

Source: https://www.youtube.com/watch?v=vnrMu4Iv7II
---------